# Deployment safety — multi-model training + Q-learning demo

Dataset: `simulation/data/traffic_simulation.csv`

- **Features:** `error_rate`, `response_time`
- **Label:** `deployment_fail` = 1 if `error_rate > 0.08` OR `response_time > 250`

**Part A:** Train several sklearn classifiers, compare accuracy / F1, save the **best** to `ml-service/model.pkl`.

**Part B:** Simple **tabular Q-learning** on a discretised traffic state — learns a policy over actions {continue, throttle, rollback} on synthetic transitions (educational; not the same as the sklearn API model).

`pip install pandas scikit-learn numpy` in the kernel if needed.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().parent
CSV_PATH = ROOT / "simulation" / "data" / "traffic_simulation.csv"
MODEL_OUT = ROOT / "ml-service" / "model.pkl"
print("CSV:", CSV_PATH.resolve())
print("Out:", MODEL_OUT.resolve())

In [ ]:
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=["error_rate", "response_time"])
df["deployment_fail"] = (
    (df["error_rate"] > 0.08) | (df["response_time"] > 250)
).astype(int)
df.head()

In [ ]:
X = df[["error_rate", "response_time"]]
y = df["deployment_fail"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "logistic_regression": LogisticRegression(max_iter=500),
    "decision_tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=80, max_depth=6, random_state=42),
    "gradient_boosting": GradientBoostingClassifier(
        n_estimators=80, max_depth=3, learning_rate=0.1, random_state=42
    ),
}

scores = []
fitted = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    acc = accuracy_score(y_test, pred)
    f1m = f1_score(y_test, pred, average="macro", zero_division=0)
    scores.append((name, acc, f1m))
    fitted[name] = m
    print(f"{name:22}  accuracy={acc:.3f}  macro_f1={f1m:.3f}")

best_name = max(scores, key=lambda t: t[2])[0]
best_model = fitted[best_name]
print("\nBest (by macro-F1):", best_name)
print(classification_report(y_test, best_model.predict(X_test), target_names=["Safe", "Fail"], zero_division=0))

### Why macro-F1?

With imbalanced Safe/Fail labels, accuracy alone can be misleading. **Macro-F1** averages F1 per class so minority "Fail" still matters.

In [ ]:
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
with open(MODEL_OUT, "wb") as f:
    pickle.dump(best_model, f)
print("Saved sklearn classifier to", MODEL_OUT)

## Part B — Tabular Q-learning (traffic policy sketch)

We **discretise** `(error_rate, response_time)` into a small state id, then simulate episodes: at each step the agent picks an action — **0 = continue**, **1 = throttle** (reduce load), **2 = rollback** (safe mode). Rewards encourage low error/latency and penalise failure conditions.

This is a **toy MDP** for coursework: the FastAPI service still uses the **sklearn** `model.pkl` above; Q-table could later drive a separate orchestration simulator.

In [ ]:
def discretize_row(er, rt, er_edges=(0.05, 0.12), rt_edges=(120, 280)):
    eb = 0 if er < er_edges[0] else 1 if er < er_edges[1] else 2
    tb = 0 if rt < rt_edges[0] else 1 if rt < rt_edges[1] else 2
    return eb * 3 + tb

n_states = 9
n_actions = 3
Q = np.zeros((n_states, n_actions))

alpha, gamma, eps = 0.35, 0.9, 0.25
n_episodes = 400

rows = df[["error_rate", "response_time", "deployment_fail"]].values

rng = np.random.default_rng(42)

for _ in range(n_episodes):
    i = int(rng.integers(0, len(rows)))
    er, rt, fail = rows[i]
    s = discretize_row(er, rt)
    if rng.random() < eps:
        a = int(rng.integers(0, n_actions))
    else:
        a = int(np.argmax(Q[s]))
    if a == 0:
        r = 1.0 - 3.0 * fail
    elif a == 1:
        r = 0.5 - 1.5 * fail
    else:
        r = 0.2 if fail else -0.3
    j = int(rng.integers(0, len(rows)))
    er2, rt2, _ = rows[j]
    s2 = discretize_row(er2, rt2)
    Q[s, a] = (1 - alpha) * Q[s, a] + alpha * (r + gamma * Q[s2].max())

policy = Q.argmax(axis=1)
action_names = ["continue", "throttle", "rollback"]
print("Learned policy (state -> action):")
for s in range(n_states):
    print(f"  state {s:2d} -> {action_names[policy[s]]}")
print("Q table sample row 0:", np.round(Q[0], 3))